### Data Cleaning and Feature Creation

In [ ]:
import pandas as pd

In [ ]:
from pickle import load

df = load(open("../../../data/interim/vbac/vbac_df.pkl", "rb"))

For simplicity, we'll create a feature called `successful_vbac` and use it as our target.

In [ ]:
df["successful_vbac"] = df.apply(
    lambda row: 1 if row["delivery_method_recode_7"] == 2 else 0, axis=1
)

df["successful_vbac"].info()

Our dataset includes many features that are irrelevant to our goal. We'll clean up our features by dropping a few different subsets of features, starting with any features that indicate that another feature has been imputed.

In [ ]:
flag_column_names = df.filter(regex="(.*imputed.*|.*flag.*)$", axis=1).columns

flag_column_names_list = flag_column_names.to_list()

We'll also remove features that are recorded after the birth of the child, such as APGAR scores and details about the particulars of the birth and recovery period.

On top of that, we've decided to exclude a few features that had a high amount of missing data, such as paternity acknowledgement.

In [ ]:
unrelated_features = [
    "birth_year",
    "paternity_acknowledged",
    "marital_status",
    "cigarette_recode",
    "fertility_enhancing_drugs",
    "assistive_reproductive_technology",
    "previous_cesarean",
    "reported_mothers_age_used",
    "final_route_and_method_of_delivery",
    "five_minute_apgar_score",
    "five_minute_apgar_recode",
    "ten_minute_apgar_score",
    "ten_minute_apgar_score_recode",
    "plurality_recode",
    "set_order_recode",
    "last_normal_menses_month",
    "last_normal_menses_year",
    "obstetric_estimate_edited",
    "assisted_ventilation_immediate",
    "assisted_ventilation_after_6_hours",
    "admission_to_nicu",
    "surfactant",
    "antibiotics_for_newborn",
    "seizures",
    "no_abnormal_conditions_reported",
    "infant_transferred",
    "infant_living_at_time_of_report",
    "infant_breastfed_at_discharge",
    "month_prenatal_care_started",
    "no_maternal_comorbidity_reported",
    "perineal_laceration",
    "ruptured_uterus",
    "total_birth_order_recode",
    "unplanned_hysterectomy",
    "maternal_transfusion",
    "mother_transferred",
    "admit_to_intensive_care",
    "live_birth_order_recode",
    "2021",
]

df_filtered = df.drop(flag_column_names_list, axis="columns").drop(
    unrelated_features, axis="columns"
)

Of the remaining features, a few have NA values, so we'll drop those rows.

In [ ]:
df_filtered = df_filtered.dropna(
    subset=[
        "WIC",
        "gonorrhea",
        "syphilis",
        "chlamydia",
        "hepatitis_b",
        "hepatitis_c",
        "successful_external_cephalic_version",
        "failed_external_cephalic_version",
        "induction_of_labor",
        "augmentation_of_labor",
        "steroids",
        "antibiotics",
        "chorioamnionitis",
        "anesthesia",
        "anencephaly",
        "spina_bifida",
        "cyanotic_congenital_heart_disease",
        "congenital_diaphragmatic_hernia",
        "omphalocele",
        "gastroschisis",
        "limb_reduction_defect",
        "cleft_lip",
        "cleft_palate",
        "down_syndrome",
        "suspected_chromosomal_disorder",
        "hypospadias",
        "no_congenital_abnormalities_reported",
    ]
)

We will impute a `0` in the `trial_of_labor_attempted_if_cesarean` feature for any missing values, as this is likely an important factor in determining whether or not a VBAC will be successful.

In [ ]:
df_filtered["trial_of_labor_attempted_if_cesarean"] = df_filtered[
    "trial_of_labor_attempted_if_cesarean"
].fillna(0)

Finally, we will extract our target value and drop any column with 1-1 correlation to it.

In [ ]:
successful_vbac = df_filtered["successful_vbac"]

df_filtered = df_filtered.drop(
    ["delivery_method_recode_7", "delivery_method_recode_3", "successful_vbac"],
    axis="columns",
)

Because successful VBAC comprises only a small portion of our populaiton, we'll stratify our train-test split.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df_filtered,
    successful_vbac,
    train_size=0.8,
    random_state=14,
    stratify=successful_vbac,
)

We'll create a categorical and a numerical pipeline for data transformation.

In [ ]:
cat_features = [
    "birth_month",
    "birth_day_of_week",
    "birth_place",
    "facility_recode",
    "mothers_age_recode_14",
    "mothers_age_recode_9",
    "mothers_nativity",
    "mothers_residence_status",
    "mothers_race_recode_31",
    "mothers_race_recode_6",
    "mothers_race_recode_15",
    "mothers_hispanic_origin",
    "mothers_hispanic_origin_recode",
    "mothers_race_and_hispanic_origin",
    "mothers_education",
    "fathers_age_recode_11",
    "fathers_race_recode_31",
    "fathers_race_recode_6",
    "fathers_race_recode_15",
    "fathers_hispanic_origin",
    "fathers_hispanic_origin_recode",
    "fathers_race_and_hispanic_origin",
    "fathers_education",
    "interval_since_last_live-birth_recode_11",
    "interval_since_last_other_pregnancy_recode",
    "interval_since_last_other_pregnancy_recode_11",
    "interval_since_last_pregnancy_recode",
    "interval_since_last_pregnancy_recode_11",
    "month_prenatal_care_began_recode",
    "number_of_prenatal_visits_recode",
    "WIC",
    "cigarettes_before_pregnancy_recode",
    "cigarettes_first_trimester_recode",
    "cigarettes_second_trimester_recode",
    "cigarettes_third_trimester_recode",
    "BMI_recode",
    "pre_pregnancy_weight_recode",
    "delivery_weight_recode",
    "weight_gain_recode",
    "pre_pregnancy_diabetes",
    "gestational_diabetes",
    "pre_pregnancy_hypertension",
    "gestational_hypertension",
    "hypertension_eclampsia",
    "previous_preterm_birth",
    "infertility_treatment_used",
    "no_risk_factors_reported",
    "gonorrhea",
    "syphilis",
    "chlamydia",
    "hepatitis_b",
    "hepatitis_c",
    "no_infection_present",
    "successful_external_cephalic_version",
    "failed_external_cephalic_version",
    "induction_of_labor",
    "augmentation_of_labor",
    "steroids",
    "antibiotics",
    "chorioamnionitis",
    "anesthesia",
    "no_characteristics_of_labor_reported",
    "fetal_presentation_at_delivery",
    "trial_of_labor_attempted_if_cesarean",
    "attendant_at_birth",
    "payment_source_for_delivery",
    "payment_recode",
    "sex_of_infant",
    "combined_gestation_recode_10",
    "combined_gestation_recode_3",
    "obstetric_estimate_recode_10",
    "obstetric_estimate_recode_3",
    "birth_weight_recode_12",
    "birth_weight_recode_4",
    "anencephaly",
    "spina_bifida",
    "cyanotic_congenital_heart_disease",
    "congenital_diaphragmatic_hernia",
    "omphalocele",
    "gastroschisis",
    "limb_reduction_defect",
    "cleft_lip",
    "cleft_palate",
    "down_syndrome",
    "suspected_chromosomal_disorder",
    "hypospadias",
    "no_congenital_abnormalities_reported",
]

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline

cat_pipeline = make_pipeline(
    SimpleImputer(missing_values=pd.NA, strategy="most_frequent"),
    OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"),
)

In [ ]:
num_features = [
    "time_of_birth",
    "mothers_single_year_age",
    "fathers_combined_age",
    "prior_births_now_living",
    "prior_births_now_dead",
    "prior_other_terminations",
    "interval_since_last_live_birth",
    "number_of_prenatal_visits",
    "cigarettes_before_pregnancy",
    "cigarettes_first_trimester",
    "cigarettes_second_trimester",
    "cigarettes_third_trimester",
    "mothers_height_in_inches",
    "BMI",
    "weight_gain",
    "number_of_previous_cesarean",
    "combined_gestation_detail_in_weeks",
    "birth_weight_in_grams",
]

In [ ]:
from sklearn.preprocessing import StandardScaler

num_pipeline = make_pipeline(
    SimpleImputer(missing_values=pd.NA, strategy="mean"), StandardScaler()
)

In [ ]:
from sklearn.compose import ColumnTransformer

preprocessing = ColumnTransformer(
    [
        ("Categorical Pipeline", cat_pipeline, cat_features),
        ("Numerical Pipeline", num_pipeline, num_features),
    ]
)

In [ ]:
X_train_prepared = preprocessing.fit_transform(X_train, y_train)
X_test_prepared = preprocessing.transform(X_test)

We now have a transformed dataset with 443,145 rows and 1603 columns.

In [ ]:
X_train_prepared.shape

We are not confident that all 1603 features will prove to be highly relevant to our model, so we will use a Random Forest for feature selection.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

feature_selector = RandomForestClassifier(random_state=14)

feature_selector.fit(X_train_prepared, y_train)

feature_importances = feature_selector.feature_importances_

In [ ]:
feature_importances_df = pd.DataFrame(
    {
        "feature": preprocessing.get_feature_names_out(),
        "importance": feature_importances,
    }
).sort_values("importance", ascending=False)

feature_importances_df.head(20)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 8))
plt.bar(range(len(feature_importances)), feature_importances)
plt.xlabel("Feature Index")
plt.ylabel("Feature Importance")
plt.title("Feature Importances from RandomForestClassifier")
plt.show()

We'll save the feature importances to a file so it can be used to filter our train and test sets in different configurations.

In [ ]:
from pickle import dump

dump(
    feature_importances_df,
    open("../../../data/processed/vbac/feature_importances_df.pkl", "wb"),
)

We'll save our processed data for model training.

In [ ]:
dump(
    X_train_prepared,
    open("../../../data/processed/vbac/X_train_prepared.pkl", "wb"),
)
dump(y_train, open("../../../data/processed/vbac/y_train.pkl", "wb"))
dump(
    X_test_prepared,
    open("../../../data/processed/vbac/X_test_prepared.pkl", "wb"),
)
dump(y_test, open("../../../data/processed/vbac/y_test.pkl", "wb"))